# Data Validation

Apply agreed business rules to the Bronze data and report records that fail.
Use the results to guide later cleaning decisions.

## Sales Validation

- Check required fields and order ID uniqueness.
- Validate quantities, amounts and dates.
- Check customer, product and store IDs against their master tables.
- Compare sales amounts with quantity multiplied by product price.
- Flag unusually large quantities for investigation.

The original Bronze tables remain unchanged.

In [0]:
from pyspark.sql import functions as F
CATALOG = "sales_quality"
SCHEMA = "bronze_layer"

#Read only columns with sales data no ingestio column 
sales = spark.table(f"{CATALOG}.{SCHEMA}.sales")

business_columns = [
    column for column in sales.columns
    if not column.startswith("_")
]

sales = sales.select(*business_columns)
total_rows = sales.count()


if total_rows == 0 :
    raise ValueError("Sales is empty. Review ingestion ")

#load master fields for sales validation. sales row may have customer,product and store id but we need the master table to confirm this data is valed 
#without loading those columns you could only validate sales internally missing vlaues and duplicate orders but we can't detect the unknown id and so on 
customers = (
    spark.table(f"{CATALOG}.{SCHEMA}.customers")
    .select("customer_id")
)
products = (
    spark.table(f"{CATALOG}.{SCHEMA}.products")
    .select("product_id", "product_price")
)
stores = (
    spark.table(f"{CATALOG}.{SCHEMA}.stores")
    .select("store_id")
)
print(f"Sales records to validate : {total_rows:,}")

Sales records to validate : 2,004


In [0]:
# Store one summary and one issue DataFrame per rule.
validation_results = {}
sales_issues = {}
def is_missing(column):
    """Identify nulls and blank text."""
    return (
        F.col(column).isNull()
        | (F.trim(F.col(column)) == "")
    )

def report_check(rule, eligible, failed, severity="ERROR"):
    """Summarize a check and retain its affected business records."""

    checked_count = eligible.count()
    failed_count = failed.count()

    # No eligible rows means the rule could not be evaluated.
    if checked_count == 0:
        status = "NOT_EVALUATED"
    elif failed_count == 0:
        status = "PASS"
    elif severity == "WARNING":
        status = "REVIEW"
    else:
        status = "FAIL"

    validation_results[rule] = (
        rule,
        severity,
        checked_count,
        checked_count - failed_count,
        failed_count,
        total_rows - checked_count,
        status,
    )

    # Keep original business values and explain the finding.
    sales_issues[rule] = (
        failed
        .select(*business_columns)
        .withColumn("failure_reason", F.lit(rule))
    )

    print(f"{rule}: {status} — {failed_count} affected rows")

    # Display a manageable sample; the DataFrame retains every failure.
    if failed_count > 0:
        display(sales_issues[rule].limit(20))


In [0]:
#fields we need to validate 
#here we validate that all rows on those columns are required
required_columns = [
    "date",
    "order_id",
    "customer_id",
    "product_id",
    "units_sold",
    "total_amount",
    "store_id",
]
#check failed columns. values on this column must be exising to validate it 
for column in required_columns:
    failed= sales.filter(is_missing(column))

    report_check(
        rule =f"{column} must be existing",
        eligible = sales,
        failed = failed
    )

date must be existing: FAIL — 3 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
null,ORD001106,CS_Mor_46,P10073,5,170,St_Pho_ A,date must be existing
null,ORD001519,CS_Far_38,P10042,2,66,St_Alb_ N,date must be existing
null,ORD000904,CS_Jar_24,P10078,3,102,St_Los_Ca,date must be existing


order_id must be existing: FAIL — 2 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2021-11-04 00:00:00,null,CS_Jon_20,P10041,3,102,St_Bal_Ma,order_id must be existing
2020-01-13 00:00:00,null,CS_Del_6,P10009,6,432,St_Sea_Wa,order_id must be existing


customer_id must be existing: FAIL — 4 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2021-04-03 00:00:00,ORD001148,null,P10049,4,128,St_Pho_ A,customer_id must be existing
2021-08-12 00:00:00,ORD001773,null,P10023,2,66,St_Mia_Fl,customer_id must be existing
2020-02-21 00:00:00,ORD000684,null,P10064,4,128,St_Col_Oh,customer_id must be existing
2020-11-11 00:00:00,ORD000844,null,P10096,2,20,St_Cha_No,customer_id must be existing


product_id must be existing: PASS — 0 affected rows
units_sold must be existing: FAIL — 4 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2020-03-21 00:00:00,ORD001059,CS_Bry_48,P10020,null,99,St_Ind_In,units_sold must be existing
2021-08-07 00:00:00,ORD001503,CS_Hun_29,P10035,null,136,St_Alb_ N,units_sold must be existing
2020-07-24 00:00:00,ORD001625,CS_Bar_13,P10022,null,124,St_Ind_In,units_sold must be existing
2021-08-02 00:00:00,ORD000077,CS_Bea_11,P10035,null,136,St_Ind_In,units_sold must be existing


total_amount must be existing: PASS — 0 affected rows
store_id must be existing: PASS — 0 affected rows


In [0]:
#validate order id is unique, on this case order id should be unique 
#missing orders validated on previous validation . so we will exclude them 
identified_sales = sales.filter(~is_missing("order_id"))

#find repated IDs 
repeated_ids = (
    identified_sales
    .groupBy("order_id")
    .count()
    .filter(F.col("count")>1)
    .select("order_id") #this shows only order_id column 
)
#display(repeated_ids)
#retrive all rows belonging to those repated IDs.
duplicated_orders = identified_sales.join(
          repeated_ids,
          on = "order_id",
          how = "inner"
)

report_check(
    rule ="order_id must be unique",
    eligible = identified_sales,
    failed= duplicated_orders
)



order_id must be unique: FAIL — 10 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2020-04-28 00:00:00,ORD001014,CS_Bel_43,P10061,6,192,St_Pho_ A,order_id must be unique
2021-06-13 00:00:00,ORD001587,CS_Stu_47,P10013,6,474,St_Cha_No,order_id must be unique
2020-10-10 00:00:00,ORD001802,CS_Sim_26,P10098,3,45,St_Col_Oh,order_id must be unique
2021-04-25 00:00:00,ORD000691,CS_Jar_24,P10053,5,175,St_Los_Ca,order_id must be unique
2020-02-19 00:00:00,ORD000805,CS_Liv_35,P10073,3,102,St_Alb_ N,order_id must be unique
2020-10-10 00:00:00,ORD001802,CS_Sim_26,P10098,3,45,St_Col_Oh,order_id must be unique
2021-06-13 00:00:00,ORD001587,CS_Stu_47,P10013,6,474,St_Cha_No,order_id must be unique
2020-04-28 00:00:00,ORD001014,CS_Bel_43,P10061,6,192,St_Pho_ A,order_id must be unique
2021-04-25 00:00:00,ORD000691,CS_Jar_24,P10053,5,175,St_Los_Ca,order_id must be unique
2020-02-19 00:00:00,ORD000805,CS_Liv_35,P10073,3,102,St_Alb_ N,order_id must be unique


In [0]:
#all numeric values are saved as string . we need to interperate them first as numeric or date on temporary columns then validate them , then report the orginal values which still string as valid or not . we don't change anything on hte original dataframe 
sales_checked  = (
    sales
    .withColumn("quantity_number",
                F.expr("try_cast(units_sold AS DOUBLE)"),)
    .withColumn("amount_number",
                F.expr("try_cast(total_amount AS DOUBLE)"),)
    .withColumn("sale_date",
                F.expr("try_cast(`date` AS DATE)"),
)
)
#let's set teh rules of valid quantity and valid amount 
#valid quanitty . should be finit, positive and whole number 
#we create bolean column called valid quantity 
valid_quantity = F.coalesce(
    F.col("quantity_number").isNotNull()
    & ~F.isnan("quantity_number")
    & (F.abs(F.col("quantity_number")) != F.lit(float("inf")))
    & (F.col("quantity_number") > 0)
    & ((F.col("quantity_number") % 1) == 0),
    F.lit(False),
)

# Amounts must be finite positive numbers under our project rules.
valid_amount = F.coalesce(
    F.col("amount_number").isNotNull()
    & ~F.isnan("amount_number")
    & (F.abs(F.col("amount_number")) != F.lit(float("inf")))
    & (F.col("amount_number") > 0),
    F.lit(False),
)


In [0]:
#now validate both quantity and amount . 
#the approach still the same .first remove missing values .then you have eligible data to check . then we define the failed data and pass both to report_check function 
quantity_candidates = sales_checked.filter(~is_missing("units_sold"))
report_check(
    rule="units_sold must be a positive whole number",
    eligible=quantity_candidates,
    failed=quantity_candidates.filter(~valid_quantity),
)

amount_candidates = sales_checked.filter(~is_missing("total_amount"))
report_check(
    rule="total_amount must be a positive number",
    eligible=amount_candidates,
    failed=amount_candidates.filter(~valid_amount),
)

units_sold must be a positive whole number: FAIL — 6 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2022-06-21 00:00:00,ORD001349,CS_Com_17,P10021,five,170,St_Det_Mi,units_sold must be a positive whole number
2021-05-04 00:00:00,ORD000291,CS_Bry_48,P10007,0,126,St_Atl_Ge,units_sold must be a positive whole number
2020-08-15 00:00:00,ORD000304,CS_Mat_12,P10054,three,96,St_Ind_In,units_sold must be a positive whole number
2020-04-22 00:00:00,ORD000592,CS_But_25,P10022,-3,124,St_Atl_Ge,units_sold must be a positive whole number
2020-07-06 00:00:00,ORD000695,CS_Jar_24,P10100,-2,60,St_Atl_Ge,units_sold must be a positive whole number
2020-08-02 00:00:00,ORD000845,CS_Har_42,P10023,0,99,St_Bal_Ma,units_sold must be a positive whole number


total_amount must be a positive number: PASS — 0 affected rows


In [0]:
# Set a consistent timezone for today's date.
spark.conf.set("spark.sql.session.timeZone", "UTC")

# we exclude missing values here . we work on date values of the original data .
date_candidates = sales_checked.filter(
    ~is_missing("date")
)

report_check(
    rule="date must be a valid calendar date",
    eligible=date_candidates,
    failed=date_candidates.filter(F.col("sale_date").isNull()),
)

# now we still have unreadable dates that will give us Null on the sales_date. so we need to exclude them as we do usually. so now thes readable column is the ones that are eligible . and otherwise . but still didn't define failed column. 
#the failed column shouldn't be null on the main date and should be readable and shoudn't be after the current date 

# Only readable dates can be compared with today.
readable_dates = date_candidates.filter(
    F.col("sale_date").isNotNull()
)


report_check(
    rule="date must not be in the future",
    eligible=readable_dates,
    failed=readable_dates.filter(
        F.col("sale_date") > F.current_date()
    ),
)

date must be a valid calendar date: FAIL — 3 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2021-13-10,ORD001895,CS_Del_4,P10027,3,90,St_Pho_ A,date must be a valid calendar date
2020-04-31,ORD000664,CS_Col_39,P10073,2,68,St_Hou_Te,date must be a valid calendar date
2022-02-30,ORD000853,CS_Har_42,P10070,2,68,St_Alb_ N,date must be a valid calendar date


date must not be in the future: FAIL — 3 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2099-01-02 00:00:00,ORD000113,CS_Kel_44,P10046,2,64,St_Sea_Wa,date must not be in the future
2099-01-01 00:00:00,ORD000188,CS_Don_33,P10090,5,85,St_Ind_In,date must not be in the future
2099-01-03 00:00:00,ORD000992,CS_Ngu_31,P10002,6,444,St_Las_Ne,date must not be in the future


In [0]:
#now we need to validate ids against the master data 
# Read Sales IDs >>> Read trusted master IDs >> Find Sales IDs with no match >> Report 

#pair each sales id with it's master table 
master_checks = [
    ("customer_id", customers),
    ("product_id", products),
    ("store_id", stores),
]

for column,master in master_checks:
    #first we choose rows where the values of itreated column is not null 
    candidates = sales.filter(~is_missing(column))
    master_ids = (
        master
        .filter(~is_missing(column)) # here we are making like validation for those columns on master tables 
        .select(column)
        .distinct()
    )

    unmatched_sales = candidates.join(
        master_ids,
        on=column,
        how="leftanti",
    )
    #it's time to report 
    report_check(
        rule=f"{column} must exist in its master table",
        eligible=candidates,
        failed=unmatched_sales,
    )


customer_id must exist in its master table: FAIL — 2 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2022-02-12 00:00:00,ORD001211,CS_UNKNOWN,P10032,4,132,St_Alb_ N,customer_id must exist in its master table
2020-09-20 00:00:00,ORD001733,CS_UNKNOWN,P10054,2,64,St_Ind_In,customer_id must exist in its master table


product_id must exist in its master table: FAIL — 2 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2020-12-31 00:00:00,ORD001300,CS_Sch_32,P99999,2,66,St_Det_Mi,product_id must exist in its master table
2021-06-17 00:00:00,ORD000537,CS_Stu_47,P99999,3,99,St_Hou_Te,product_id must exist in its master table


store_id must exist in its master table: FAIL — 2 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2021-06-18 00:00:00,ORD000253,CS_Dav_37,P10033,4,140,ST_UNKNOWN,store_id must exist in its master table
2020-03-27 00:00:00,ORD000554,CS_Har_42,P10016,3,96,ST_UNKNOWN,store_id must exist in its master table


In [0]:
#Validate the calculated sales amount
# the approach is get sales product_id then find it's product price from products table then converts units sold to number and product_price to number as well and multiply them then compare with total_amount which needs to be converted to number as well 

#we convert product_price on products table to numeric
product_prices = products.withColumn(
    "reference_price",
    F.expr("try_cast(product_price AS DOUBLE)",)
)
#let's check if there is any products have more than one id 
# we work on original vaues  
ambiguous_prices = (
    product_prices
    .filter(~is_missing("product_id"))
    .groupBy("product_id")
    .count()
    .filter(F.col("count")>1)
)

if ambiguous_prices.count()>0 : 
    raise ValueError(
        "products contain repated IDs.")


#we need to keep only finite positive values for the producted converted prices 
product_prices = (
    product_prices
    .filter(
        F.col("reference_price").isNotNull()
        & ~F.isnan("reference_price")
        & (F.abs(F.col("reference_price")) != F.lit(float("inf"))) #reject positive and negative infinity 
        & (F.col("reference_price") > 0) # keep only positive values. Zero is rejected as well 
    )
    .select("product_id", "reference_price") # we dodn't need the main price column any more 
)


# Evaluate totals only when the necessary inputs are usable.
amount_comparison = (
    sales_checked
    .filter(valid_quantity & valid_amount) # this one rely on that we have defined column names on their origin clumn expression 
    .join(product_prices, on="product_id", how="inner")  
    .withColumn(
        "expected_amount",
        F.col("quantity_number") * F.col("reference_price"),
    )
)

incorrect_totals = amount_comparison.filter(
    F.abs(
        F.col("amount_number") - F.col("expected_amount")
    ) >= 0.005
)

report_check(
    rule="total_amount must equal quantity multiplied by product price",
    eligible=amount_comparison,
    failed=incorrect_totals,
)

# Show the calculation behind each mismatch.
display(
    incorrect_totals.select(
        "order_id",
        "product_id",
        "units_sold",
        "reference_price",
        "total_amount",
        "expected_amount",
    )
)


total_amount must equal quantity multiplied by product price: FAIL — 5 affected rows


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,failure_reason
2020-03-20 00:00:00,ORD000052,CS_Leo_14,P10037,4,153,St_Ind_In,total_amount must equal quantity multiplied by product price
2021-10-07 00:00:00,ORD000170,CS_Tur_45,P10034,2,85,St_Las_Ne,total_amount must equal quantity multiplied by product price
2021-06-07 00:00:00,ORD000290,CS_Smi_10,P10095,5,75,St_Det_Mi,total_amount must equal quantity multiplied by product price
2022-01-03 00:00:00,ORD000329,CS_Mar_40,P10094,4,73,St_Los_Ca,total_amount must equal quantity multiplied by product price
2021-08-28 00:00:00,ORD000583,CS_Dav_37,P10010,5,415,St_Los_Ca,total_amount must equal quantity multiplied by product price


order_id,product_id,units_sold,reference_price,total_amount,expected_amount
ORD000052,P10037,4,32.0,153,128.0
ORD000170,P10034,2,30.0,85,60.0
ORD000290,P10095,5,10.0,75,50.0
ORD000329,P10094,4,12.0,73,48.0
ORD000583,P10010,5,78.0,415,390.0


In [0]:
#display all validation summary 
# Build the final summary from all recorded rule results.
sales_validation_summary = spark.createDataFrame(
    list(validation_results.values()),
    """
    rule string,
    severity string,
    checked_rows long,
    passed_rows long,
    affected_rows long,
    not_evaluated_rows long,
    status string
    """,
)

display(
    sales_validation_summary.orderBy(
        F.desc("affected_rows"),
        "rule",
    )
)

rule,severity,checked_rows,passed_rows,affected_rows,not_evaluated_rows,status
order_id must be unique,ERROR,2002,1992,10,2,FAIL
units_sold must be a positive whole number,ERROR,2000,1994,6,4,FAIL
total_amount must equal quantity multiplied by product price,ERROR,1992,1987,5,12,FAIL
customer_id must be existing,ERROR,2004,2000,4,0,FAIL
units_sold must be existing,ERROR,2004,2000,4,0,FAIL
date must be a valid calendar date,ERROR,2001,1998,3,3,FAIL
date must be existing,ERROR,2004,2001,3,0,FAIL
date must not be in the future,ERROR,1998,1995,3,6,FAIL
customer_id must exist in its master table,ERROR,2000,1998,2,4,FAIL
order_id must be existing,ERROR,2004,2002,2,0,FAIL


## Customers Validation

- Check that all required customer fields are populated.
- Verify that each populated customer ID appears only once.
- Display records that fail, with clear reasons.
- Produce a summary of passed, failed and unevaluated checks.

The results will support later cleaning decisions. No customer records are changed or removed.

In [0]:
customers = spark.table(f"{CATALOG}.{SCHEMA}.customers")
# Select business columns only.
customer_columns = [
    column for column in customers.columns
    if not column.startswith("_")
]

customers = customers.select(*customer_columns)

customer_total_rows = customers.count()

if customer_total_rows == 0:
    raise ValueError("Customers is empty. Review ingestion first.")

# Store customer rule summaries and affected records.
customer_validation_results = []
customer_issues = {}

print(f"Customer records to validate: {customer_total_rows}")

Customer records to validate: 49


In [0]:
#we would start by definig the required columns . We will use the same columns as in the original table
# Required fields under this project's customer rules.
required_customer_columns = [
    "customer_id",
    "first_name",
    "last_name",
    "gender",
    "city",
]

for column in required_customer_columns:

    rule = f"{column} must be populated"

    # Find records with a missing required value.
    failed = customers.filter(is_missing(column))
    failed_count = failed.count()

    # Every customer record is eligible for completeness checks.
    passed_count = customer_total_rows - failed_count
    status = "PASS" if failed_count == 0 else "FAIL"

    customer_validation_results.append((
        rule,
        customer_total_rows,
        passed_count,
        failed_count,
        0,  # No records are excluded from this check.
        status,
    ))

    # Retain every failed record with its reason.
    customer_issues[rule] = failed.withColumn(
        "failure_reason",
        F.lit(rule),
    )

    print(f"{rule}: {status} — {failed_count} affected rows")

    # Show examples only when failures exist.
    if failed_count > 0:
        display(customer_issues[rule].limit(20))

customer_id must be populated: PASS — 0 affected rows
first_name must be populated: PASS — 0 affected rows
last_name must be populated: PASS — 0 affected rows
gender must be populated: PASS — 0 affected rows
city must be populated: PASS — 0 affected rows


In [0]:
# now we will do same thing with customer id uniqueness 

# exclude null values 
identified_customers = customers.filter(
    ~is_missing("customer_id")
)

# Find customer IDs appearing more than once.
repeated_customer_ids = (
    identified_customers
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .select("customer_id")
)

# Retrieve all records belonging to repeated IDs.
duplicate_customers = (
    identified_customers
    .join(
        repeated_customer_ids,
        on="customer_id",
        how="inner",
    )
    .select(*customer_columns)
)

checked_count = identified_customers.count()
failed_count = duplicate_customers.count()
passed_count = checked_count - failed_count

# Missing IDs are not evaluated for uniqueness.
not_evaluated_count = customer_total_rows - checked_count

if checked_count == 0:
    status = "NOT_EVALUATED"
elif failed_count == 0:
    status = "PASS"
else:
    status = "FAIL"

rule = "customer_id must be unique"

customer_validation_results.append((
    rule,
    checked_count,
    passed_count,
    failed_count,
    not_evaluated_count,
    status,
))

# Retain duplicate records for inspection.
customer_issues[rule] = duplicate_customers.withColumn(
    "failure_reason",
    F.lit(rule),
)

print(f"{rule}: {status} — {failed_count} affected rows")

if failed_count > 0:
    display(
        customer_issues[rule]
        .orderBy("customer_id")
        .limit(20)
    )

customer_id must be unique: PASS — 0 affected rows


In [0]:
# Create one summary row per customer validation rule.
customer_validation_summary = spark.createDataFrame(
    customer_validation_results,
    """
    rule string,
    checked_rows long,
    passed_rows long,
    affected_rows long,
    not_evaluated_rows long,
    status string
    """,
)

display(
    customer_validation_summary.orderBy(
        F.desc("affected_rows"),
        "rule",
    )
)

rule,checked_rows,passed_rows,affected_rows,not_evaluated_rows,status
city must be populated,49,49,0,0,PASS
customer_id must be populated,49,49,0,0,PASS
customer_id must be unique,49,49,0,0,PASS
first_name must be populated,49,49,0,0,PASS
gender must be populated,49,49,0,0,PASS
last_name must be populated,49,49,0,0,PASS


## Products Validation

- Check that required product fields are populated.
- Verify that each populated product ID is unique.
- Require product prices to be finite positive numbers.
- Compare product categories with the approved category reference.
- Show known category corrections without applying them.
- Produce a validation summary and a report of affected records.

No product records are changed or removed.

In [0]:
# Read the full Products table from Bronze.
products = spark.table(f"{CATALOG}.{SCHEMA}.products")

# Exclude ingestion metadata.
product_columns = [
    column for column in products.columns
    if not column.startswith("_")
]

products = products.select(*product_columns)
product_total_rows = products.count()

if product_total_rows == 0:
    raise ValueError("Products is empty. Review ingestion first.")

#We will use refernces tables for the validation of products
# Load the approved category values.
approved_categories = (
    spark.table(f"{CATALOG}.{SCHEMA}.product_categories")
    .select("category_name")
)

# Load known spelling variations and their approved corrections.
category_aliases = (
    spark.table(f"{CATALOG}.{SCHEMA}.category_aliases")
    .select("raw_value", "correct_value")
)

print(f"Product records to validate: {product_total_rows}")

Product records to validate: 100


In [0]:
#we build same helper functions 
# Separate containers for Products validation.
product_validation_results = {}
product_issues = {}

def report_product_check(rule, eligible, failed):
    """Record a product rule's results and affected records."""

    checked_count = eligible.count()
    failed_count = failed.count()

    if checked_count == 0:
        status = "NOT_EVALUATED"
    elif failed_count == 0:
        status = "PASS"
    else:
        status = "FAIL"

    # Store one summary per rule.
    product_validation_results[rule] = (
        rule,
        checked_count,
        checked_count - failed_count,
        failed_count,
        product_total_rows - checked_count,
        status,
    )

    # Keep original business values and add the failure reason.
    product_issues[rule] = (
        failed
        .select(*product_columns)
        .withColumn("failure_reason", F.lit(rule))
    )

    print(f"{rule}: {status} — {failed_count} affected rows")

    if failed_count > 0:
        display(product_issues[rule].limit(20))

In [0]:
#we would start by validating required fields 
required_product_columns = [
    "product_id",
    "product_name",
    "product_price",
    "product_category",
]

for column in required_product_columns:

    # Find missing values in the current required field.
    failed = products.filter(is_missing(column))

    report_product_check(
        rule=f"{column} must be populated",
        eligible=products,
        failed=failed,
    )

product_id must be populated: PASS — 0 affected rows
product_name must be populated: PASS — 0 affected rows
product_price must be populated: PASS — 0 affected rows
product_category must be populated: PASS — 0 affected rows


In [0]:
#Validate product ID uniqueness
# Missing IDs are covered by the completeness check.
identified_products = products.filter(
    ~is_missing("product_id")
)
# Find populated IDs used by multiple records.
repeated_product_ids = (
    identified_products
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .select("product_id")
)

# Retrieve every record belonging to a repeated ID.
duplicate_products = identified_products.join(
    repeated_product_ids,
    on="product_id",
    how="inner",
)

report_product_check(
    rule="product_id must be unique",
    eligible=identified_products,
    failed=duplicate_products,
)

product_id must be unique: PASS — 0 affected rows


In [0]:
#Validate product prices
#convert first product_price into number 
product_price_data = products.withColumn(
    "parsed_price",
    F.expr("try_cast(product_price AS DOUBLE)"),
)
#now we need to exlude values failed to be converted 
price_candidates = product_price_data.filter(
    ~is_missing("product_price")
) #now those are eligible values . but we need to validate them 

# Reject conversion failures, NaN, infinity, zero and negatives.
#first set the rule 
valid_price = F.coalesce(
    F.col("parsed_price").isNotNull()
    & ~F.isnan("parsed_price")
    & (F.abs(F.col("parsed_price")) != F.lit(float("inf")))
    & (F.col("parsed_price") > 0),
    F.lit(False),
)
#second apply it to get failed ones 
invalid_prices = price_candidates.filter(~valid_price)

report_product_check(
    rule="product_price must be a finite positive number",
    eligible=price_candidates,
    failed=invalid_prices,
)

product_price must be a finite positive number: PASS — 0 affected rows


In [0]:
#Validate categories against the approved reference
# Prepare the approved category lookup.
category_lookup = (
    approved_categories
    .filter(~is_missing("category_name"))
    .select(
        F.col("category_name").alias("product_category")
    )
    .distinct()
)

# Missing categories are already covered by completeness.
category_candidates = products.filter(
    ~is_missing("product_category")
)

# Find categories absent from the approved list.
invalid_product_categories = category_candidates.join(
    category_lookup,
    on="product_category",
    how="left_anti",
)

report_product_check(
    rule="product_category must match an approved category",
    eligible=category_candidates,
    failed=invalid_product_categories,
)

product_category must match an approved category: FAIL — 3 affected rows


product_id,product_name,product_price,product_category,failure_reason
P10001,Air purifier,66,electronics,product_category must match an approved category
P10014,oven gloves,35,Unknown Category,product_category must match an approved category
P10043,Ankle Training Socks,31,Clothng,product_category must match an approved category


In [0]:
#Show whether failed categories have known corrections
#rename names of reference table 
alias_lookup = category_aliases.select(
    F.col("raw_value").alias("product_category"),
    F.col("correct_value").alias("suggested_category"),
)


# Attach known corrections to failed product categories.
product_category_review = (
    invalid_product_categories
    .join(
        alias_lookup,
        on="product_category",
        how="left",
    )
    .withColumn(
        "review_note",
        F.when(
            F.col("suggested_category").isNotNull(),
            "Known alias: correction available",
        ).otherwise(
            "No approved alias: review required"
        ),
    )
    .select(
        "product_id",
        "product_name",
        "product_category",
        "suggested_category",
        "review_note",
    )
)

display(
    product_category_review.orderBy("product_id")
)

product_id,product_name,product_category,suggested_category,review_note
P10001,Air purifier,electronics,Electronics,Known alias: correction available
P10014,oven gloves,Unknown Category,null,No approved alias: review required
P10043,Ankle Training Socks,Clothng,Clothing,Known alias: correction available


In [0]:
#display validation summary for the whole products table 
product_validation_summary = spark.createDataFrame(
    list(product_validation_results.values()),
    """
    rule string,
    checked_rows long,
    passed_rows long,
    affected_rows long,
    not_evaluated_rows long,
    status string
    """,
)

display(
    product_validation_summary.orderBy(
        F.desc("affected_rows"),
        "rule",
    )
)

rule,checked_rows,passed_rows,affected_rows,not_evaluated_rows,status
product_category must match an approved category,100,97,3,0,FAIL
product_category must be populated,100,100,0,0,PASS
product_id must be populated,100,100,0,0,PASS
product_id must be unique,100,100,0,0,PASS
product_name must be populated,100,100,0,0,PASS
product_price must be a finite positive number,100,100,0,0,PASS
product_price must be populated,100,100,0,0,PASS


## Stores Validation

- Check that required store fields are populated.
- Verify that each populated store ID is unique.
- Compare city, state and location together with the approved reference.
- Display records that fail, with clear reasons.
- Produce a validation summary and an issues report.

No store records are changed or removed.

In [0]:
#Load Stores and the location reference
# Read the Stores table from Bronze.
stores = spark.table(f"{CATALOG}.{SCHEMA}.stores")

# Exclude ingestion metadata.
store_columns = [
    column for column in stores.columns
    if not column.startswith("_")
]

stores = stores.select(*store_columns)
store_total_rows = stores.count()

if store_total_rows == 0:
    raise ValueError("Stores is empty. Review ingestion first.")

# Load approved location combinations.
store_locations = (
    spark.table(f"{CATALOG}.{SCHEMA}.store_locations")
    .select("city", "state", "location")
)

print(f"Store records to validate: {store_total_rows}")

Store records to validate: 14


In [0]:
#Prepare store validation reporting. build same helper functions 
# Store summaries and affected records separately for Stores.
store_validation_results = {}
store_issues = {}

def report_store_check(rule, eligible, failed):
    """Record a store rule's results and affected records."""

    checked_count = eligible.count()
    failed_count = failed.count()

    if checked_count == 0:
        status = "NOT_EVALUATED"
    elif failed_count == 0:
        status = "PASS"
    else:
        status = "FAIL"

    # Keep one summary per rule.
    store_validation_results[rule] = (
        rule,
        checked_count,
        checked_count - failed_count,
        failed_count,
        store_total_rows - checked_count,
        status,
    )

    # Keep original business values with a failure reason.
    store_issues[rule] = (
        failed
        .select(*store_columns)
        .withColumn("failure_reason", F.lit(rule))
    )

    print(f"{rule}: {status} — {failed_count} affected rows")

    if failed_count > 0:
        display(store_issues[rule].limit(20))

In [0]:
#start by validation required fields 
required_store_columns = [
    "store_id",
    "city",
    "state",
    "location",
]

for column in required_store_columns:

    # Find records missing the current required value.
    failed = stores.filter(is_missing(column))

    report_store_check(
        rule=f"{column} must be populated",
        eligible=stores,
        failed=failed,
    )

store_id must be populated: PASS — 0 affected rows
city must be populated: PASS — 0 affected rows
state must be populated: PASS — 0 affected rows
location must be populated: PASS — 0 affected rows


In [0]:
#Validate store ID uniqueness
# Keep records eligible for the uniqueness check.
identified_stores = stores.filter(
    ~is_missing("store_id")
)

# Find IDs appearing more than once.
repeated_store_ids = (
    identified_stores
    .groupBy("store_id")
    .count()
    .filter(F.col("count") > 1)
    .select("store_id")
)

# Retrieve every record belonging to a repeated ID.
duplicate_stores = identified_stores.join(
    repeated_store_ids,
    on="store_id",
    how="inner",
)

report_store_check(
    rule="store_id must be unique",
    eligible=identified_stores,
    failed=duplicate_stores,
)

store_id must be unique: PASS — 0 affected rows


In [0]:
#Validate the complete location combination
#we need to check if the store three values are valid together City+State+Location
#Read all soters >> Exclude rows with missing city,state or location >> compare all three colummns with store_locations refrence table >>Report those stores as Failures

# Keep records with all three location fields populated.
location_candidates = stores.filter(
    ~is_missing("city")
    & ~is_missing("state")
    & ~is_missing("location")
)


# Prepare complete, distinct approved combinations.
location_lookup = (
    store_locations
    .filter(
        ~is_missing("city")
        & ~is_missing("state")
        & ~is_missing("location")
    )
    .distinct()
)

# Find store combinations without an exact reference match.
invalid_store_locations = location_candidates.join(
    location_lookup,
    on=["city", "state", "location"], #join on all three columns 
    how="left_anti",
)

report_store_check(
    rule="City, state and location must match an approved combination",
    eligible=location_candidates,
    failed=invalid_store_locations,
)

City, state and location must match an approved combination: FAIL — 2 affected rows


store_id,city,state,location,failure_reason
St_Atl_Ge,Atlanta,Florida,East Coast,"City, state and location must match an approved combination"
St_Mia_Fl,Miami,Florida,Midwest,"City, state and location must match an approved combination"


In [0]:
# Show reference values beside the failures
# Rename reference fields so we can compare them with stored values.
expected_locations = location_lookup.select(
    "city",
    F.col("state").alias("reference_state"),
    F.col("location").alias("reference_location"),
)

# Keep all failures, including cities absent from the reference.
store_location_review = (
    invalid_store_locations
    .join(
        expected_locations,
        on="city",
        how="left",
    )
    .withColumn(
        "review_note",
        F.when(
            F.col("reference_state").isNull(),
            "City absent from reference: review required",
        ).otherwise(
            "Compare stored values with the approved combination"
        ),
    )
    .select(
        "store_id",
        "city",
        "state",
        "reference_state    ",
        "location",
        "reference_location",
        "review_note",
    )
)

display(
    store_location_review.orderBy("city", "store_id")
)

store_id,city,state,reference_state,location,reference_location,review_note
St_Atl_Ge,Atlanta,Florida,Georgia,East Coast,East Coast,Compare stored values with the approved combination
St_Mia_Fl,Miami,Florida,Florida,Midwest,East Coast,Compare stored values with the approved combination


In [0]:
#Display the Stores validation summary
store_validation_summary = spark.createDataFrame(
    list(store_validation_results.values()),
    """
    rule string,
    checked_rows long,
    passed_rows long,
    affected_rows long,
    not_evaluated_rows long,
    status string
    """,
)

display(
    store_validation_summary.orderBy(
        F.desc("affected_rows"),
        "rule",
    )
)

rule,checked_rows,passed_rows,affected_rows,not_evaluated_rows,status
"City, state and location must match an approved combination",14,12,2,0,FAIL
city must be populated,14,14,0,0,PASS
location must be populated,14,14,0,0,PASS
state must be populated,14,14,0,0,PASS
store_id must be populated,14,14,0,0,PASS
store_id must be unique,14,14,0,0,PASS
